# Python 3.14：FastAPI + React 前后端分离网站开发


## 1. 学习目标

课程结束时，学生能够：

1. 区分前端、后端与 API。
2. 使用 FastAPI 编写 `GET`、`POST`、`DELETE` 接口。
3. 使用 `/docs` 测试 API。
4. 理解 JSON、HTTP Method、Status Code 和 CORS。
5. 使用 React 的 `useState` 保存页面数据。
6. 使用 `useEffect` 在页面打开后加载数据。
7. 使用 `fetch()` 调用 FastAPI。
8. 完成一个小型前后端分离网站。


# 第一部分：前后端分离是什么？



# 第二部分：创建项目

## 1. 检查 Python 3.14

## 2. 创建目录

```bash
mkdir fastapi-react-course
cd fastapi-react-course
mkdir backend
```

## 3. 创建 Python 3.14 虚拟环境

```bash
cd backend
python3.14 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install "fastapi[standard]"
```

看到 Terminal 提示符前出现 `(.venv)`，说明激活成功。


## 4. 创建 React 项目

回到项目根目录：

```bash
cd ..
npm create vite@latest frontend -- --template react
cd frontend
npm install
cd ..
```

项目结构：

```text
fastapi-react-course/
├── backend/
│   ├── .venv/
│   └── main.py
└── frontend/
    ├── src/
    │   ├── App.jsx
    │   ├── App.css
    │   └── main.jsx
    └── package.json
```
> `backend` 和 `frontend` 是两个独立程序，所以稍后要开两个 Terminal 分别启动。


# 第三部分：HTTP、API 与 JSON

## API 地址的组成

```text
http://127.0.0.1:8000/api/tasks
└协议┘ └──主机──┘└端口┘└─路径─┘
```

## 本课使用的 HTTP Method

| Method | 含义 | 本课示例 |
|---|---|---|
| `GET` | 读取数据 | 获取任务列表 |
| `POST` | 创建数据 | 添加一个任务 |
| `DELETE` | 删除数据 | 删除指定任务 |

## JSON 示例

```json
{
  "id": 3,
  "title": "学习 React"
}
```

JSON 与 Python 字典很像，但要注意：JSON 的字符串和属性名使用双引号。

## 常用状态码

- `200 OK`：读取成功
- `201 Created`：创建成功
- `204 No Content`：删除成功，不返回正文
- `404 Not Found`：没有找到数据
- `422 Unprocessable Content`：提交的数据不符合规则


# 第四部分：课堂演示：FastAPI 后端

在 `backend` 目录创建 `main.py`。下面是完整代码。



In [1]:
# 文件：backend/main.py
# 这是课堂演示代码。真正启动服务时，请把它保存到 main.py，
# 然后在 Terminal 中运行：fastapi dev main.py

from fastapi import FastAPI, HTTPException, Response, status
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field


app = FastAPI(
    title="学习任务 API",
    description="Python 3.14 + FastAPI 课堂演示",
    version="1.0.0",
)


# React 和 FastAPI 端口不同，浏览器会把它们视为不同 Origin。
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173",
    ],
    allow_methods=["GET", "POST", "DELETE"],
    allow_headers=["Content-Type"],
)


# 前端创建任务时提交的数据格式
class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=50)


# 后端返回给前端的数据格式
class Task(TaskCreate):
    id: int


# 课堂演示暂时用列表保存数据；后端重启后会恢复初始值。
tasks: list[dict] = [
    {"id": 1, "title": "学习 Python 基础"},
    {"id": 2, "title": "认识 FastAPI 接口"},
]

next_id = 3


@app.get("/")
def home():
    return {"message": "学习任务 API 正在运行"}


@app.get("/api/tasks", response_model=list[Task])
def get_tasks():
    # 返回全部任务
    return tasks


@app.post(
    "/api/tasks",
    response_model=Task,
    status_code=status.HTTP_201_CREATED,
)
def create_task(task: TaskCreate):
    # 创建一个新任务
    global next_id

    clean_title = task.title.strip()
    if not clean_title:
        raise HTTPException(
            status_code=422,
            detail="任务内容不能为空",
        )

    new_task = {
        "id": next_id,
        "title": clean_title,
    }
    tasks.append(new_task)
    next_id += 1
    return new_task


@app.delete("/api/tasks/{task_id}")
def delete_task(task_id: int):
    # 根据 ID 删除一个任务
    for index, task in enumerate(tasks):
        if task["id"] == task_id:
            tasks.pop(index)
            return Response(
                status_code=status.HTTP_204_NO_CONTENT
            )

    raise HTTPException(
        status_code=404,
        detail="没有找到这个任务",
    )


## 后端代码逐段讲解

### 1. 创建应用

```python
app = FastAPI()
```

可以理解为“创建一个后端网站对象”。后面的路由都注册到 `app` 上。

### 2. Pydantic Model 是数据表格

```python
class TaskCreate(BaseModel):
    title: str = Field(min_length=1, max_length=50)
```

它告诉 FastAPI：前端必须提交字符串 `title`，长度为 1～50。

### 3. 路由装饰器

```python
@app.get("/api/tasks")
def get_tasks():
    return tasks
```

`@app.get(...)` 可以读成：“当收到这个地址的 GET 请求时，执行下面的函数。”

### 4. 动态路径参数

```python
@app.delete("/api/tasks/{task_id}")
```

`/api/tasks/2` 中的 `2` 会自动成为函数参数 `task_id=2`。

### 5. 为什么用 `list[Task]`

Python 3.14 可以直接使用现代类型标注：

```python
response_model=list[Task]
```

它表示接口返回的是一个任务列表，并让 Swagger UI 显示清晰的数据结构。


## 启动 FastAPI

新开或切换到后端 Terminal：

```bash
cd fastapi-react-course/backend
source .venv/bin/activate
python --version
fastapi dev main.py
```

确认 `python --version` 显示 Python 3.14.x。

浏览器访问：

- 后端首页：<http://127.0.0.1:8000>
- Swagger UI：<http://127.0.0.1:8000/docs>

首页预期返回：

```json
{"message": "学习任务 API 正在运行"}
```


# 第五部分：用 `/docs` 测试 API

## 测试 GET

1. 展开 `GET /api/tasks`。
2. 点击 **Try it out**。
3. 点击 **Execute**。
4. 找到 Status Code `200` 和 Response Body。

## 测试 POST

展开 `POST /api/tasks`，提交：

```json
{
  "title": "学习 React useState"
}
```

预期状态码为 `201`。

## 测试 DELETE

展开 `DELETE /api/tasks/{task_id}`，输入任务 ID，例如 `1`。

预期状态码为 `204`。

### 课堂快速提问

- `POST` 为什么不是 `GET`？
- 删除一个不存在的 ID 会发生什么？
- 提交空字符串时，为什么会看到 `422`？


## CORS：为什么后端要写“允许名单”？

React 地址：

```text
http://localhost:5173
```

FastAPI 地址：

```text
http://127.0.0.1:8000
```

端口不同就是不同 Origin。浏览器默认不会随便允许网页读取另一个 Origin 的数据，因此后端需要配置 CORS。

```python
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],
    allow_methods=["GET", "POST", "DELETE"],
    allow_headers=["Content-Type"],
)
```

零基础类比：

> CORS 是后端门口的访客名单。React 的地址在名单里，浏览器才允许它读取后端响应。


# 第六部分：编写 React 网页

## 先认识三个 React 工具

### `useState`：组件的记忆

```jsx
const [tasks, setTasks] = useState([]);
```

- `tasks`：当前任务列表。
- `setTasks`：修改任务列表并让页面重新显示。
- `[]`：初始值为空数组。

### `useEffect`：页面显示后执行操作

```jsx
useEffect(() => {
  loadTasks();
}, []);
```

本例中表示：页面第一次显示后，从 FastAPI 读取一次数据。

### `fetch`：向后端发送请求

```jsx
const response = await fetch(API_URL);
const data = await response.json();
setTasks(data);
```


## 完整 React 代码：`frontend/src/App.jsx`

```jsx
import { useEffect, useState } from "react";
import "./App.css";

const API_URL = "http://127.0.0.1:8000/api/tasks";

function App() {
  const [tasks, setTasks] = useState([]);
  const [title, setTitle] = useState("");
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState("");

  useEffect(() => {
    loadTasks();
  }, []);

  async function loadTasks() {
    try {
      setError("");
      const response = await fetch(API_URL);

      if (!response.ok) {
        throw new Error("获取任务失败");
      }

      const data = await response.json();
      setTasks(data);
    } catch (err) {
      setError("无法连接后端，请检查 FastAPI 是否已启动。");
    } finally {
      setLoading(false);
    }
  }

  async function addTask(event) {
    event.preventDefault();
    const cleanTitle = title.trim();

    if (!cleanTitle) {
      setError("请先输入任务内容。");
      return;
    }

    try {
      setError("");
      const response = await fetch(API_URL, {
        method: "POST",
        headers: {
          "Content-Type": "application/json",
        },
        body: JSON.stringify({ title: cleanTitle }),
      });

      if (!response.ok) {
        throw new Error("添加任务失败");
      }

      const newTask = await response.json();
      setTasks((current) => [...current, newTask]);
      setTitle("");
    } catch (err) {
      setError("添加失败，请稍后重试。");
    }
  }

  async function deleteTask(taskId) {
    try {
      setError("");
      const response = await fetch(`${API_URL}/${taskId}`, {
        method: "DELETE",
      });

      if (!response.ok) {
        throw new Error("删除任务失败");
      }

      setTasks((current) =>
        current.filter((task) => task.id !== taskId)
      );
    } catch (err) {
      setError("删除失败，请稍后重试。");
    }
  }

  return (
    <main className="page">
      <section className="task-card">
        <header className="page-header">
          <p className="eyebrow">FASTAPI + REACT</p>
          <h1>我的学习任务</h1>
          <p className="subtitle">记录今天最重要的学习目标</p>
        </header>

        <form className="task-form" onSubmit={addTask}>
          <input
            type="text"
            value={title}
            maxLength={50}
            placeholder="例如：练习 FastAPI GET 接口"
            onChange={(event) => setTitle(event.target.value)}
          />
          <button type="submit">添加任务</button>
        </form>

        {error && <p className="error-message">{error}</p>}

        {loading ? (
          <p className="status-message">正在读取任务……</p>
        ) : (
          <ul className="task-list">
            {tasks.map((task) => (
              <li className="task-item" key={task.id}>
                <span className="task-number">#{task.id}</span>
                <span className="task-title">{task.title}</span>
                <button
                  className="delete-button"
                  type="button"
                  onClick={() => deleteTask(task.id)}
                >
                  删除
                </button>
              </li>
            ))}
          </ul>
        )}
      </section>
    </main>
  );
}

export default App;
```


## 完整网页样式：`frontend/src/App.css`

```css
:root {
  font-family: Inter, "PingFang SC", "Microsoft YaHei", sans-serif;
  color: #172033;
  background: #eef2ff;
}

* { box-sizing: border-box; }
body { margin: 0; min-width: 320px; min-height: 100vh; }
button, input { font: inherit; }
button { cursor: pointer; }

.page {
  min-height: 100vh;
  padding: 64px 20px;
  background:
    radial-gradient(circle at top left, rgba(129, 140, 248, .35), transparent 35%),
    linear-gradient(135deg, #f8fafc, #eef2ff);
}

.task-card {
  width: min(680px, 100%);
  margin: 0 auto;
  padding: 38px;
  border: 1px solid rgba(148, 163, 184, .25);
  border-radius: 24px;
  background: rgba(255, 255, 255, .94);
  box-shadow: 0 24px 70px rgba(30, 41, 59, .14);
}

.page-header { margin-bottom: 28px; }
.eyebrow {
  margin: 0 0 8px;
  color: #4f46e5;
  font-size: 13px;
  font-weight: 800;
  letter-spacing: .13em;
}
h1 { margin: 0; font-size: clamp(32px, 7vw, 48px); }
.subtitle { margin: 12px 0 0; color: #64748b; font-size: 17px; }

.task-form { display: flex; gap: 12px; }
.task-form input {
  flex: 1;
  min-width: 0;
  padding: 14px 16px;
  border: 1px solid #cbd5e1;
  border-radius: 12px;
  outline: none;
}
.task-form input:focus {
  border-color: #6366f1;
  box-shadow: 0 0 0 4px rgba(99, 102, 241, .14);
}
.task-form button {
  padding: 14px 20px;
  border: 0;
  border-radius: 12px;
  color: white;
  background: #4f46e5;
  font-weight: 700;
}
.task-form button:hover { background: #4338ca; }

.task-list {
  display: grid;
  gap: 12px;
  margin: 28px 0 0;
  padding: 0;
  list-style: none;
}
.task-item {
  display: flex;
  align-items: center;
  gap: 12px;
  padding: 15px;
  border: 1px solid #e2e8f0;
  border-radius: 14px;
  background: #f8fafc;
}
.task-number { color: #6366f1; font-size: 13px; font-weight: 800; }
.task-title { flex: 1; }
.delete-button {
  padding: 7px 12px;
  border: 0;
  border-radius: 9px;
  color: #be123c;
  background: #ffe4e6;
}
.delete-button:hover { color: white; background: #e11d48; }
.error-message {
  padding: 12px 14px;
  border-radius: 10px;
  color: #b91c1c;
  background: #fee2e2;
}
.status-message { margin-top: 28px; color: #64748b; text-align: center; }

@media (max-width: 560px) {
  .page { padding: 24px 12px; }
  .task-card { padding: 24px 18px; }
  .task-form { flex-direction: column; }
}
```


## 启动 React

再打开一个 Terminal：

```bash
cd fastapi-react-course/frontend
npm run dev
```

打开：<http://localhost:5173>

此时应同时保留两个服务：

```text
Terminal 1：FastAPI  → http://127.0.0.1:8000
Terminal 2：React    → http://localhost:5173
```

1. 刷新网页时，浏览器发送了哪种请求？
2. 点击“添加任务”时，发送的是 `GET` 还是 `POST`？
3. React 得到新任务后，哪一行代码更新了网页？
4. 为什么每个任务需要唯一的 `id`？


## Notebook 内网页效果预览

运行下一格，可以在 Notebook 中看到最终网页的静态交互预览。这个预览使用浏览器内 JavaScript 模拟添加和删除；正式课堂项目仍使用 React + FastAPI。


In [3]:
from IPython.display import HTML, display

display(HTML(r"""
<div id="course-demo" style="font-family:-apple-system,BlinkMacSystemFont,'PingFang SC',sans-serif;padding:42px 16px;background:linear-gradient(135deg,#f8fafc,#eef2ff);border-radius:20px">
  <section style="max-width:620px;margin:auto;padding:32px;border:1px solid #e2e8f0;border-radius:22px;background:white;box-shadow:0 20px 50px rgba(30,41,59,.12)">
    <p style="color:#4f46e5;font-size:12px;font-weight:800;letter-spacing:.13em;margin:0 0 8px">FASTAPI + REACT</p>
    <h1 style="font-size:38px;margin:0;color:#172033">我的学习任务</h1>
    <p style="color:#64748b;margin:10px 0 24px">Notebook 交互效果预览</p>
    <div style="display:flex;gap:10px">
      <input id="demo-title" placeholder="例如：练习 FastAPI GET 接口" style="flex:1;padding:13px;border:1px solid #cbd5e1;border-radius:11px">
      <button onclick="addDemoTask()" style="padding:13px 18px;border:0;border-radius:11px;color:white;background:#4f46e5;font-weight:700">添加任务</button>
    </div>
    <p id="demo-count" style="color:#64748b;margin:22px 0 10px"></p>
    <ul id="demo-list" style="display:grid;gap:10px;padding:0;list-style:none"></ul>
  </section>
</div>
<script>
  let demoTasks = [
    {id: 1, title: '学习 Python 基础'},
    {id: 2, title: '认识 FastAPI 接口'}
  ];
  let demoNextId = 3;
  function renderDemoTasks() {
    const list = document.getElementById('demo-list');
    const count = document.getElementById('demo-count');
    count.innerHTML = `当前共有 <strong style="color:#4f46e5">${demoTasks.length}</strong> 个任务`;
    list.innerHTML = demoTasks.length ? demoTasks.map(task => `
      <li style="display:flex;align-items:center;gap:10px;padding:13px;border:1px solid #e2e8f0;border-radius:12px;background:#f8fafc">
        <span style="color:#6366f1;font-size:12px;font-weight:800">#${task.id}</span>
        <span style="flex:1;color:#172033">${task.title}</span>
        <button onclick="deleteDemoTask(${task.id})" style="padding:6px 10px;border:0;border-radius:8px;color:#be123c;background:#ffe4e6">删除</button>
      </li>`).join('') : '<li style="padding:20px;text-align:center;color:#64748b;border:1px dashed #cbd5e1;border-radius:12px">暂时没有任务，添加第一个目标吧。</li>';
  }
  function addDemoTask() {
    const input = document.getElementById('demo-title');
    const title = input.value.trim();
    if (!title) return;
    demoTasks.push({id: demoNextId++, title});
    input.value = '';
    renderDemoTasks();
  }
  function deleteDemoTask(id) {
    demoTasks = demoTasks.filter(task => task.id !== id);
    renderDemoTasks();
  }
  document.getElementById('demo-title').addEventListener('keydown', event => {
    if (event.key === 'Enter') addDemoTask();
  });
  renderDemoTasks();
</script>
"""))


# 第七部分：联调与排错

在 Chrome 中按 `Command + Option + I`，打开 **Network** 面板。

依次操作并观察：

| 操作 | Method | 预期状态码 |
|---|---|---|
| 刷新网页 | GET | 200 |
| 添加任务 | POST | 201 |
| 删除任务 | DELETE | 204 |

## 高频错误清单

### 1. `Failed to fetch`

- FastAPI 是否仍在运行？
- `API_URL` 是否写成 `http://127.0.0.1:8000/api/tasks`？

### 2. `Blocked by CORS policy`

- React 地址是否在 `allow_origins` 中？
- 修改 `main.py` 后，后端是否已重新加载？

### 3. `npm: command not found`

Node.js 未安装，或安装后 Terminal 尚未重开。

### 4. 后端重启后新增任务消失

当前数据保存在 Python 内存列表中，不是数据库。这是正常现象。


# 第八部分：——总结

## 请学生补全数据流

```text
点击添加按钮
  → React 的 __________ 函数执行
  → fetch 发送 __________ 请求
  → FastAPI 的 create_task() 执行
  → FastAPI 返回 __________ 数据
  → React 调用 __________ 更新任务数组
  → 页面重新渲染
```

参考答案：`addTask`、`POST`、`JSON`、`setTasks`。

## 五句课程总结

1. React 负责页面与交互。
2. FastAPI 负责接口、数据与业务规则。
3. 前后端通过 HTTP 和 JSON 通信。
4. React 使用 State 保存会变化的页面数据。
5. 前后端地址不同，需要正确配置 CORS。


# 课后作业：增加“完成状态”和筛选功能

## 作业目标

在课堂项目基础上继续开发，让每个任务可以标记为已完成。

### 必做功能

1. 每个任务增加 `done` 字段，初始值为 `false`。
2. 增加 `PATCH /api/tasks/{task_id}` 接口。
3. 网页可以切换“已完成/未完成”。
4. 已完成任务显示删除线。
5. 增加三个筛选按钮：全部、未完成、已完成。
6. 显示总任务数、已完成数和未完成数。

### 新任务数据示例

```json
{
  "id": 3,
  "title": "完成课后作业",
  "done": false
}
```

### PATCH 请求示例

```http
PATCH /api/tasks/3
Content-Type: application/json
```

```json
{
  "done": true
}
```


## 课后作业：后端提示

新增 Model：

```python
class TaskUpdate(BaseModel):
    done: bool
```

为 `Task` 增加字段：

```python
class Task(TaskCreate):
    id: int
    done: bool = False
```

创建新任务时增加：

```python
new_task = {
    "id": next_id,
    "title": clean_title,
    "done": False,
}
```

PATCH 接口参考结构：

```python
@app.patch("/api/tasks/{task_id}", response_model=Task)
def update_task(task_id: int, update: TaskUpdate):
    for task in tasks:
        if task["id"] == task_id:
            task["done"] = update.done
            return task

    raise HTTPException(
        status_code=404,
        detail="没有找到这个任务",
    )
```

CORS 中加入 `PATCH`：

```python
allow_methods=["GET", "POST", "DELETE", "PATCH"]
```


## 课后作业：前端提示

### 切换完成状态

```jsx
async function toggleTask(task) {
  const response = await fetch(`${API_URL}/${task.id}`, {
    method: "PATCH",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ done: !task.done }),
  });

  const updatedTask = await response.json();

  setTasks((current) =>
    current.map((item) =>
      item.id === updatedTask.id ? updatedTask : item
    )
  );
}
```

### 统计数量

```jsx
const completedCount = tasks.filter((task) => task.done).length;
const unfinishedCount = tasks.length - completedCount;
```

### 完成样式

```jsx
<span className={task.done ? "task-title completed" : "task-title"}>
  {task.title}
</span>
```

```css
.completed {
  color: #94a3b8;
  text-decoration: line-through;
}
```

### 筛选思路

```jsx
const [filter, setFilter] = useState("all");

const visibleTasks = tasks.filter((task) => {
  if (filter === "done") return task.done;
  if (filter === "todo") return !task.done;
  return true;
});
```


## 课后作业提交与评分

### 提交内容

- `backend/main.py`
- `frontend/src/App.jsx`
- `frontend/src/App.css`
- 一张网页运行截图
- 一段不超过 100 字的排错记录

### 评分标准

| 项目 | 分值 |
|---|---:|
| FastAPI PATCH 接口正确 | 25 |
| React 可切换完成状态 | 25 |
| 全部/未完成/已完成筛选正确 | 20 |
| 三种数量统计正确 | 15 |
| 页面样式与错误提示 | 10 |
| 提交完整 | 5 |
| **总分** | **100** |

### 加分项

- 使用 SQLite，让后端重启后数据仍然存在。
- 增加“清除已完成”按钮。
- 增加任务创建时间。
- 增加编辑任务名称功能。
